# 文書からキーワードを抽出する (埋め込みによる KeyBERT 方式)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/11_topics_keywords/keyword_extraction.ipynb)

NLP クックブック 11-03 ／ トピックとキーワードの抽出

1 つの文書から、その内容を代表するキーワードを数個選びたい。コーパス全体の統計なしで、単独の文書に対して行いたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import numpy as np
from fugashi import Tagger
from sentence_transformers import SentenceTransformer

tagger = Tagger()
model = SentenceTransformer("intfloat/multilingual-e5-small")

def candidates(text):
    """連続する名詞をつなげた複合名詞を候補にする"""
    out, buf = [], []
    for w in list(tagger(text)) + [None]:
        if w is not None and w.feature.pos1 in ("名詞", "接頭辞", "接尾辞") and w.feature.pos2 != "数詞":
            buf.append(w.surface)
        else:
            if buf and len("".join(buf)) > 1:
                out.append("".join(buf))
            buf = []
    return list(dict.fromkeys(out))

def keywords(text, top_k=6, diversity=0.5):
    cands = candidates(text)
    doc = model.encode("query: " + text, normalize_embeddings=True)
    emb = model.encode(["query: " + c for c in cands], normalize_embeddings=True)
    sim_doc, sim_cand = emb @ doc, emb @ emb.T
    selected = [int(sim_doc.argmax())]
    while len(selected) < min(top_k, len(cands)):      # MMR: 文書に近く、かつ選択済みの語とは似ていないものを選ぶ
        rest = [i for i in range(len(cands)) if i not in selected]
        mmr = [(1 - diversity) * sim_doc[i] - diversity * sim_cand[i, selected].max() for i in rest]
        selected.append(rest[int(np.argmax(mmr))])
    return [(cands[i], round(float(sim_doc[i]), 3)) for i in selected]

text = ("文法誤り訂正は、学習者が書いた文に含まれる誤りを自動的に訂正する自然言語処理のタスクである。"
        "近年は系列変換モデルや大規模言語モデルを用いた手法が主流となっており、擬似誤りデータによる事前学習が性能向上に寄与している。"
        "評価には編集単位の適合率と再現率に基づく F0.5 が広く用いられる。")
print("候補:", candidates(text))
print("キーワード:", keywords(text))

候補: ['文法誤り訂正', '学習者', '誤り', '自動的', '訂正', '自然言語処理', 'タスク', '近年', '系列変換モデル', '大規模言語モデル', '手法', '主流', '擬似誤りデータ', '事前学習', '性能向上', '寄与', '評価', '編集単位', '適合率', '再現率']
キーワード: [('文法誤り訂正', 0.899), ('大規模言語モデル', 0.85), ('タスク', 0.826), ('再現率', 0.835), ('事前学習', 0.838), ('編集単位', 0.835)]


- 「文書全体の埋め込みに近い語句ほど、その文書を代表している」という考え方。学習もコーパスも要らない。
- MMR で多様性を持たせないと、「言語モデル」「大規模言語モデル」のように似た語ばかりが選ばれる。
- 統計ベースなら TF-IDF (この章の「TF-IDF で文書ごとの特徴語を取り出す」を参照)、英語なら YAKE や RAKE、グラフベースなら TextRank などがある。ライブラリは `keybert` や `pke`。